# GRPO-train an LLM portfolio manager

Post-trains **Qwen3-4B-Instruct** with **GRPO** (via Unsloth + TRL) to allocate a weekly ETF portfolio.

- **Prompt**: market-state table (returns, vol, 52w position) for 12 ETFs on one historical day
- **Completion**: brief reasoning + JSON weights
- **Reward**: realized forward 5-day log return of that allocation (invalid output = -1)
- GRPO samples several allocations for the *same day*, so advantages are relative — market direction cancels out.

**Runtime: A100 GPU** (Runtime > Change runtime type). L4 works with `load_in_4bit=True` and smaller batch.

In [ ]:
%%capture
!pip install unsloth vllm
!pip install yfinance pyarrow

In [ ]:
# Get the project code + data
REPO_URL = "https://github.com/YOUR_USERNAME/rl-finance.git"  # <-- set me

import os, sys
if not os.path.exists("rl-finance"):
    !git clone {REPO_URL} rl-finance
sys.path.insert(0, "rl-finance/src")

from rl_finance.data import read_jsonl
from rl_finance.rewards import grpo_reward_func
from pathlib import Path

DATA = Path("rl-finance/data")
train_samples = read_jsonl(DATA / "train.jsonl")
val_samples = read_jsonl(DATA / "val.jsonl")
print(len(train_samples), "train prompts |", len(val_samples), "val prompts")
print(train_samples[0]["prompt"][:600])

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,        # set True on L4/T4
    fast_inference=True,       # vLLM backend for fast GRPO sampling
    max_lora_rank=32,
    gpu_memory_utilization=0.7,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import Dataset

def to_row(s):
    return {
        "prompt": [{"role": "user", "content": s["prompt"]}],
        "fwd_returns": s["fwd_returns"],
        "date": s["date"],
    }

train_ds = Dataset.from_list([to_row(s) for s in train_samples])
train_ds = train_ds.shuffle(seed=3407)
train_ds

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="outputs",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_generations=8,           # group size per market day
    max_prompt_length=1280,
    max_completion_length=256,
    temperature=1.0,
    num_train_epochs=2,
    logging_steps=5,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[grpo_reward_func],  # gets `fwd_returns` column via kwargs
    args=config,
    train_dataset=train_ds,
)
trainer.train()

## Out-of-sample evaluation (validation split, 2020-2022)

Generate one allocation per validation day at low temperature, then run the
same backtester used for the benchmarks.

In [ ]:
from vllm import SamplingParams
from rl_finance.rewards import parse_weights

sampling = SamplingParams(temperature=0.2, max_tokens=256)

model.save_lora("grpo_trader_lora")
lora_req = model.load_lora("grpo_trader_lora")

texts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": s["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    for s in val_samples
]
outputs = model.fast_generate(texts, sampling_params=sampling, lora_request=lora_req)

decisions = {}
n_invalid = 0
for s, out in zip(val_samples, outputs):
    w = parse_weights(out.outputs[0].text)
    if w is None:
        n_invalid += 1
        w = {"CASH": 1.0}
    decisions[s["date"]] = w
print(f"{n_invalid}/{len(val_samples)} invalid outputs")

In [ ]:
import pandas as pd
from rl_finance.backtest import run_backtest
from rl_finance.benchmarks import BENCHMARKS
from rl_finance.data import SPLITS, download_prices, DATA_DIR
from rl_finance.metrics import format_table

prices = pd.read_parquet(DATA_DIR / "prices.parquet") if (DATA_DIR / "prices.parquet").exists() else download_prices()

def llm_policy(date, feats):
    # Dates with no generated decision (shouldn't happen mid-sample) sit in cash.
    return decisions.get(str(date.date()), {"CASH": 1.0})

start, end = SPLITS["val"]
results = {"llm_agent": run_backtest(llm_policy, prices, start, end)["metrics"]}
for name, policy in BENCHMARKS.items():
    results[name] = run_backtest(policy, prices, start, end)["metrics"]
print(format_table(results))

In [ ]:
# Save the LoRA adapter (mount Drive to persist across sessions)
model.save_lora("grpo_trader_lora")

# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r grpo_trader_lora /content/drive/MyDrive/